In [ ]:
# Project root setup
import os
import sys
from pathlib import Path

ROOT = next((path for path in (Path.cwd(), *Path.cwd().parents) if (path / "src").is_dir()), None)
if ROOT is None:
    raise RuntimeError("Could not locate the project root containing src/.")
os.chdir(ROOT)
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))


In [1]:
import numpy as np
from numpy import linalg as la
from time import perf_counter
import os
from joblib import Parallel, delayed

import itertools

from src.model import Nonneg_dagma, MetMulDagma, MetMulColide
import src.utils as utils

PATH = './results/tuning/'
PATH_SACHS = './datasets/sachs/'
SEED = 10
N_CPUS = os.cpu_count() // 2
np.random.seed(SEED)

DATASET = "SACHS"  # SYNTH, SACHS

In [2]:
def get_lamb_value(n_nodes, n_samples, times=1):
    return np.sqrt(np.log(n_nodes) / n_samples) * times 

def cartesian_product(hyperparams):
    """
    Generate all combinations of hyperparameters.
    """
    param_names = list(hyperparams.keys())
    param_values = list(hyperparams.values())    
    param_combinations = [dict(zip(param_names, values)) for values in itertools.product(*param_values)]
    
    return param_combinations

args2str = lambda arguments: ''.join([f'{key[:3]}={val} ' for key, val in arguments.items()])

def print_best(key, metrics, args_combs, agg_funct='mean', all_best=False):
    agg_metric = {key: getattr(np, agg_funct)(value, axis=0) for key, value in metrics.items()}
        
    best_value = np.min(agg_metric[key])
    best_idxs = np.where(agg_metric[key] == best_value)[0]

    if not all_best:
        best_idxs = [best_idxs[0]]

    print(f'Combination{"s" if all_best else ""} with best {key} (agg: {agg_funct}):')
    for idx in best_idxs:
        print(args_combs[idx])
        print(f'shd: {agg_metric["shd"][idx]:.2f} | err: {agg_metric["err"][idx]:.4f} |' +
              f'acyc: {agg_metric["acyc"][idx]:.6f} | time: {agg_metric["time"][idx]:.2f}')
        
    # idx = np.argmin(agg_metric[key])

    # print(f'Combination with best {key} (agg: {agg_funct}):')
    # print(args_combs[idx])
    # print(f'shd: {agg_metric["shd"][idx]:.2f} | err: {agg_metric["err"][idx]:.4f} |' +
    #       f'acyc: {agg_metric["acyc"][idx]:.6f} | time: {agg_metric["time"][idx]:.2f}')



def run_grid_search_tuning(g, data_p, args_combs, model_const, model_args, std_x, fix_lamb, thr, verb=False):
    # Create data
    if DATASET == "SACHS":
        W_true = np.load(PATH_SACHS + "sachs_A_matrix.npy")
        X = np.load(PATH_SACHS + "sachs_X.npy")
    else:
        W_true, _, X = utils.simulate_sem(**data_p)
        
    # X = X/np.linalg.norm(X, axis=1, keepdims=True) if std_x else X
    X = utils.standarize(X) if std_x else X
    norm_W_true = np.linalg.norm(W_true)
    W_true_bin = utils.to_bin(W_true, thr)
    M, N = X.shape

    fidelity = 1/data_p['n_samples']*la.norm(X - X @ W_true, 'fro')**2

    print(f'Graph {g+1}: Fidelity: {fidelity:.3f}')

    shd, err, acyc, runtime = [np.zeros((len(args_combs)))  for _ in range(4)]
    for i, args in enumerate(args_combs):
        args_aux = args.copy()
        args_aux['lamb'] = args_aux['lamb'] if fix_lamb else get_lamb_value(N, M, args['lamb'])
        model = model_const(**model_args)
        t_init = perf_counter()
        if isinstance(model, MetMulColide):
            W_est, _ = model.fit(X, **args_aux)
        else:
            W_est = model.fit(X, **args_aux)
        t_end = perf_counter()
    
        W_est_bin = utils.to_bin(W_est, thr)
        shd[i], _, _ = utils.count_accuracy(W_true_bin, W_est_bin)
        err[i] = utils.compute_norm_sq_err(W_true, W_est, norm_W_true)
        runtime[i] = t_end - t_init
        acyc[i] = model.dagness(W_est)

        if verb and g == 0:
            text = args2str(args)
            print(f'\t- {text}: shd {shd[i]}  -  err: {err[i]:.3f}  -  acyc: {acyc[i]:.5g}  -  time: {runtime[i]:.3f}')
    
    return shd, err, acyc, runtime

## Experiment parameters

In [ ]:
model_const = MetMulColide  # MetMulDagma

model_args = {
    'primal_opt': 'sca',  # 'adam', 'fista', 'pgd
    'acyclicity': 'logdet',
    'restart': True,  # Only used in FISTA
}

verb = True
thr = .2
n_dags = 30 if DATASET != "SACHS" else 1
std_x = False
fix_lamb = False
N = 100  
data_params = {
    'n_nodes': N,
    'n_samples': 500, # 1000,
    'graph_type': 'er',
    'edges': 4*N,
    'edge_type': 'positive',
    'w_range': (.5, 1),
    'var': 1, # 1/np.sqrt(N),
}

Hyperparams = {
    'stepsize': [1e-5, 5e-5, 1e-4, 3e-4], # 5e-5
    'alpha_0': [.01],
    'rho_0': [.05],
    'beta': [1.2, 2], # higher?
    's': [1],
    'lamb': [.1],
    'iters_in': [500, 1000, 5000, 10000, 20000], #[30000, 50000],
    'iters_out': [5, 10, 25, 50],
    'tol': [1e-6],
    'sca_adam': [True],
}

# Hyperparams = {
#     'stepsize': [5e-5], # 5e-5
#     'alpha_0': [.1],
#     'rho_0': [.05],
#     'beta': [2], # higher?
#     's': [1],
#     'lamb': [.1],
#     'iters_in': [20000], #[30000, 50000],
#     'iters_out': [10],
#     'tol': [1e-6],
#     'sca_adam': [True],
# }

if DATASET == "SACHS":
    N_CPUS = 1

print('CPUs employed:', N_CPUS)
print('Looking hyperparameters for dataset', DATASET)
# Get combination of hyperparams for grid search
args_combs = cartesian_product(Hyperparams)    

t_init = perf_counter()
results = Parallel(n_jobs=N_CPUS)(delayed(run_grid_search_tuning)
                  (g, data_params, args_combs, model_const, model_args, std_x, fix_lamb, thr, verb) for g in range(n_dags))
t_end = perf_counter()

shd, err, acyc, runtime = zip(*results)
metrics = {'shd': shd, 'err': err, 'acyc': acyc, 'time': runtime}

CPUs employed: 64
Looking hyperparameters for dataset SYNTH
Graph 1: Fidelity: 100.161
Graph 11: Fidelity: 100.009
Graph 2: Fidelity: 100.284
Graph 5: Fidelity: 99.723
Graph 9: Fidelity: 99.669
Graph 6: Fidelity: 100.394
Graph 23: Fidelity: 99.139
Graph 10: Fidelity: 99.881
Graph 25: Fidelity: 100.752
Graph 22: Fidelity: 99.123
Graph 8: Fidelity: 100.955
Graph 28: Fidelity: 100.270
Graph 12: Fidelity: 99.691
Graph 24: Fidelity: 99.260
Graph 7: Fidelity: 99.326
Graph 13: Fidelity: 99.781
Graph 14: Fidelity: 100.608
Graph 17: Fidelity: 99.816
Graph 3: Fidelity: 99.626
Graph 19: Fidelity: 98.432
Graph 15: Fidelity: 100.315
Graph 30: Fidelity: 100.232
Graph 27: Fidelity: 100.163
Graph 4: Fidelity: 99.734
Graph 16: Fidelity: 99.703
Graph 21: Fidelity: 100.056
Graph 26: Fidelity: 99.735
Graph 29: Fidelity: 99.440
Graph 18: Fidelity: 100.467
Graph 20: Fidelity: 98.821
	- ste=1e-05 alp=0.01 rho=0.01 bet=1.5 s=1 lam=0.1 ite=500 ite=25 tol=1e-06 : shd 54.0  -  err: 0.133  -  acyc: 0.00068813  - 

In [4]:
print_best('shd', metrics, args_combs)
print_best('err', metrics, args_combs, all_best=True)
print()
print_best('shd', metrics, args_combs, agg_funct='median')
print_best('err', metrics, args_combs, agg_funct='median', all_best=True)


NameError: name 'metrics' is not defined

In [11]:
leg = [args2str(args) for args in args_combs]
utils.display_results(leg, metrics, agg='mean', file_name=f'{PATH}tuning_mean')
utils.display_results(leg, metrics, agg='median', file_name=f'{PATH}tuning_med')

,leg,shd,err,acyc,time
0,ste=1e-05 alp=0.01 rho=0.01 bet=1.5 s=1 lam=0....,97.2667 ± 74.4907,0.2282 ± 0.1943,0.0011 ± 0.0013,7.4856 ± 3.5356
1,ste=1e-05 alp=0.01 rho=0.01 bet=1.5 s=1 lam=0....,89.9667 ± 78.0463,0.2161 ± 0.2002,0.0000 ± 0.0000,6.6711 ± 2.5409
2,ste=1e-05 alp=0.01 rho=0.01 bet=1.5 s=1 lam=0....,181.4333 ± 156.4195,0.6683 ± 0.7326,0.0000 ± 0.0000,6.6427 ± 3.0740
3,ste=1e-05 alp=0.01 rho=0.01 bet=1.5 s=1 lam=0....,51.0333 ± 57.3605,0.1130 ± 0.1039,0.0020 ± 0.0035,4.8411 ± 1.6013
4,ste=1e-05 alp=0.01 rho=0.01 bet=1.5 s=1 lam=0....,44.1000 ± 53.4680,0.1014 ± 0.0981,0.0000 ± 0.0000,6.0283 ± 2.6535
5,ste=1e-05 alp=0.01 rho=0.01 bet=1.5 s=1 lam=0....,95.2333 ± 134.1975,0.3262 ± 0.5746,0.0000 ± 0.0000,8.1797 ± 4.4422
6,ste=1e-05 alp=0.01 rho=0.01 bet=1.5 s=1 lam=0....,0.3000 ± 0.5859,0.0116 ± 0.0035,0.0021 ± 0.0019,19.9190 ± 4.6040
7,ste=1e-05 alp=0.01 rho=0.01 bet=1.5 s=1 lam=0....,0.2667 ± 0.5735,0.0112 ± 0.0019,0.0000 ± 0.0000,25.9731 ± 4.0728
8,ste=1e-05 alp=0.01 rho=0.01 bet=1.5 s=1 lam=0....,82.7333 ± 165.0531,0.3647 ± 0.7079,0.0000 ± 0.0000,29.8041 ± 5.8817
9,ste=1e-05 alp=0.01 rho=0.01 bet=1.5 s=1 lam=0....,0.6000 ± 1.3808,0.0119 ± 0.0035,0.0020 ± 0.0022,39.5371 ± 7.2943


DataFrame saved to ./results/tuning/tuning_mean.csv


,leg,shd,err,acyc,time
0,ste=1e-05 alp=0.01 rho=0.01 bet=1.5 s=1 lam=0....,60.0000 ± 74.4907,0.1267 ± 0.1943,0.0005 ± 0.0013,7.1879 ± 3.5356
1,ste=1e-05 alp=0.01 rho=0.01 bet=1.5 s=1 lam=0....,48.5000 ± 78.0463,0.1131 ± 0.2002,0.0000 ± 0.0000,6.7499 ± 2.5409
2,ste=1e-05 alp=0.01 rho=0.01 bet=1.5 s=1 lam=0....,150.0000 ± 156.4195,0.3545 ± 0.7326,0.0000 ± 0.0000,6.6226 ± 3.0740
3,ste=1e-05 alp=0.01 rho=0.01 bet=1.5 s=1 lam=0....,22.0000 ± 57.3605,0.0609 ± 0.1039,0.0007 ± 0.0035,4.9287 ± 1.6013
4,ste=1e-05 alp=0.01 rho=0.01 bet=1.5 s=1 lam=0....,13.5000 ± 53.4680,0.0427 ± 0.0981,0.0000 ± 0.0000,5.6524 ± 2.6535
5,ste=1e-05 alp=0.01 rho=0.01 bet=1.5 s=1 lam=0....,64.5000 ± 134.1975,0.1428 ± 0.5746,0.0000 ± 0.0000,6.6489 ± 4.4422
6,ste=1e-05 alp=0.01 rho=0.01 bet=1.5 s=1 lam=0....,0.0000 ± 0.5859,0.0108 ± 0.0035,0.0021 ± 0.0019,20.1009 ± 4.6040
7,ste=1e-05 alp=0.01 rho=0.01 bet=1.5 s=1 lam=0....,0.0000 ± 0.5735,0.0108 ± 0.0019,0.0000 ± 0.0000,26.7347 ± 4.0728
8,ste=1e-05 alp=0.01 rho=0.01 bet=1.5 s=1 lam=0....,0.0000 ± 165.0531,0.0117 ± 0.7079,0.0000 ± 0.0000,29.5927 ± 5.8817
9,ste=1e-05 alp=0.01 rho=0.01 bet=1.5 s=1 lam=0....,0.0000 ± 1.3808,0.0106 ± 0.0035,0.0018 ± 0.0022,39.7444 ± 7.2943


DataFrame saved to ./results/tuning/tuning_med.csv


In [ ]:
# ### Load and plot data
# import pandas as pd
# pd.set_option("display.max_colwidth", None)

# df_mean = pd.read_csv(PATH + "tuning_mean.csv")
# df_med = pd.read_csv(PATH + "tuning_med.csv")


# df_mean["err_num"] = (
#     df_mean["err"]
#       .str.extract(r"([0-9]+\.?[0-9]*)")[0]
#       .astype(float)
# )
# print("Args with min mean err:")
# print(df_mean.loc[df_mean["err_num"].idxmin()])
# print()

# df_med["err_num"] = (
#     df_med["err"]
#       .str.extract(r"([0-9]+\.?[0-9]*)")[0]
#       .astype(float)
# )
# print("Args with min median err:")
# print(df_med.loc[df_med["err_num"].idxmin()])


Args with min mean err:
leg        ste=5e-06 alp=0.01 rho=5 bet=2 s=1 lam=0.01 ite=500 ite=10 tol=1e-06 
shd                                                            17.0000  ± 0.0000
err                                                             0.8243  ± 0.0000
acyc                                                            0.0000  ± 0.0000
time                                                            0.4262  ± 0.0000
err_num                                                                   0.8243
Name: 408, dtype: object

Args with min median err:
leg        ste=5e-06 alp=0.01 rho=5 bet=2 s=1 lam=0.01 ite=500 ite=10 tol=1e-06 
shd                                                            17.0000  ± 0.0000
err                                                             0.8243  ± 0.0000
acyc                                                            0.0000  ± 0.0000
time                                                            0.4262  ± 0.0000
err_num                          